In [3]:
extracted_v1 = {name: ExtractedDate(**d) for name, d in '{"distribution_date": {"source_page": 1, "original_text": "Distributed on Budget Day: 16 February 2024", "date_text": "16 February 2024", "normalized_date": "2024-02-16"}, "estate_duty_date": {"source_page": 36, "original_text": "Estate Duty does not apply to a person who dies after 15 February 2008.", "date_text": "15 February 2008", "normalized_date": "2008-02-15"}}' and json.loads("{\"distribution_date\": {\"source_page\": 1, \"original_text\": \"Distributed on Budget Day: 16 February 2024\", \"date_text\": \"16 February 2024\", \"normalized_date\": \"2024-02-16\"}, \"estate_duty_date\": {\"source_page\": 36, \"original_text\": \"Estate Duty does not apply to a person who dies after 15 February 2008.\", \"date_text\": \"15 February 2008\", \"normalized_date\": \"2008-02-15\"}}").items()}

## 0. Setup

Two new dependencies on top of Part 1's `requirements.txt`:

```bash
pip install mcp==1.30.0 langchain-mcp-adapters==0.3.2
```

| Package | Why |
|---|---|
| `mcp` | The official MCP Python SDK — both the server (`FastMCP`) and the client (`ClientSession`). |
| `langchain-mcp-adapters` | Turns MCP tools into LangChain tools, so they plug into the same LangChain + Gemini stack Part 1 uses. |

**Design decision — `mcp` is pinned to 1.30.0, not the latest 2.x.** `langchain-mcp-adapters` 0.3.2
requires `mcp<2.0.0`. A dry-run install showed these two packages only *add* dependencies
(starlette, uvicorn, sse-starlette, ...) and upgrade nothing Part 1 pins.

In [1]:
import json, re, socket, subprocess, sys, time
from datetime import datetime
from pathlib import Path

import pdfplumber

PDF_PATH = Path("data/fy2024_analysis_of_revenue_and_expenditure.pdf")
assert PDF_PATH.exists(), f"Document not found at {PDF_PATH}"

# Same parser as Part 1 (chosen in part_1.ipynb §1.4), keyed by printed page number (verified in §1.1).
with pdfplumber.open(PDF_PATH) as pdf:
    PAGES: dict[int, str] = {i + 1: (page.extract_text() or "") for i, page in enumerate(pdf.pages)}

Path("mcp_servers").mkdir(exist_ok=True)   # MCP server scripts live here
print(f"Parsed {len(PAGES)} pages")

Parsed 37 pages


## 1. Look at the source dates first

Before designing a tool, check what it will actually be given: which date strings exist on the two
cited pages, in what format, and what else nearby could be mistaken for them.

In [2]:
MONTHS = "January|February|March|April|May|June|July|August|September|October|November|December"
DATE = rf"\d{{1,2}} (?:{MONTHS}) \d{{4}}"

for n in (1, 36):
    print(f"page {n}:")
    for line in PAGES[n].splitlines():
        if re.search(DATE, line):
            print("   ", line.strip())

page 1:
    Distributed on Budget Day: 16 February 2024
page 36:
    person who dies after 15 February 2008. Lifelong Learning Endowment Fund, and
    2024 is from 1 April 2024 to 31 March 2025.


**Observations:**

| Page | Sentence (read by hand from the PDF) | Expected ISO date |
|---|---|---|
| 1 | "Distributed on Budget Day: 16 February 2024" | `2024-02-16` |
| 36 | "Estate Duty does not apply to a person who dies after 15 February 2008." | `2008-02-15` |

1. **Both dates use the same format, `D Month YYYY`.** A datetime tool doesn't need a fuzzy parser for this document.
2. **Page 36 has a distractor.** The Financial Year definition ("1 April 2024 to 31 March 2025") is on the same page but has nothing to do with estate duty. So the extraction step has to find the right date; it can't just take every date on the page.

## 2. Exploring MCP

### 2.1 What MCP is, in one paragraph

The **Model Context Protocol** is a client–server protocol that lets an
application find and call *tools* hosted in a separate process. There are three roles:

- **Server** — exposes tools (name, description, JSON input schema) and runs them.
- **Client** — connects to a server, lists its tools, and sends tool calls.
- **Host** — the application that holds the LLM. Here that's this notebook.

Key point: **the LLM never speaks MCP.** The host reads the server's tool schemas, passes them to the
LLM as ordinary function declarations, and when the LLM asks for a call, the host forwards it through
the client. So MCP changes *where a tool lives and how it's discovered*, not how function calling works.

### 2.2 The ways to do it

There are three independent choices. For each row, you pick **one** option:

| Choice | Options |
|---|---|
| **Transport** — how client and server talk | **stdio**: the client starts the server as a subprocess and talks over stdin/stdout. No port, and the server lives only as long as the client needs it. <br>**Streamable HTTP**: the server is a long-running HTTP process at a URL. Many clients can share it, and it can be remote. <br>*SSE (Server-Sent Events)*: the older HTTP transport, deprecated in the MCP spec in favour of Streamable HTTP. Skipped. |
| **Server library** | `FastMCP` inside the official `mcp` SDK (decorator-based) · the standalone `fastmcp` package (adds auth, proxying, server composition) · the SDK's low-level `Server` (hand-written handlers) |
| **How the agent gets the tool** | **raw `mcp.ClientSession`**: the SDK's low-level client; you list tools and send calls yourself (§2.4) · **`langchain-mcp-adapters`**: a wrapper that opens a `ClientSession` for you and turns each MCP tool into a LangChain tool (§2.5–2.6) |

Both stdio and HTTP-on-localhost count as "local MCP". The probes below try each client/transport
combination on the same one-tool server, so they can be compared on something real.

### 2.3 A probe server

A single tool, used only for exploration. `FastMCP` builds the tool's JSON schema from the type hints
and docstring, so the docstring is effectively the prompt the LLM will see.

In [3]:
%%writefile mcp_servers/probe_server.py
"""Exploration-only MCP server: one tool, so the transports can be compared on something real."""
import sys
from datetime import datetime

from mcp.server.fastmcp import FastMCP

mcp = FastMCP("probe")


@mcp.tool()
def parse_to_iso(text: str, fmt: str) -> str:
    """Parse `text` with the strptime format `fmt` and return the date as YYYY-MM-DD."""
    return datetime.strptime(text, fmt).date().isoformat()


if __name__ == "__main__":
    # `python probe_server.py` -> stdio.  `python probe_server.py http 8123` -> Streamable HTTP on localhost:8123
    if sys.argv[1:2] == ["http"]:
        mcp.settings.port = int(sys.argv[2])
        mcp.run(transport="streamable-http")
    else:
        mcp.run(transport="stdio")

Overwriting mcp_servers/probe_server.py


### 2.4 stdio + raw `ClientSession`

The lowest level: start the server over stdio, then do the protocol handshake, list the tools and call one by hand.
`sys.executable` makes the server run in the same venv as this notebook.

In [4]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

PROBE = StdioServerParameters(command=sys.executable, args=["mcp_servers/probe_server.py"])


async def raw_probe():
    async with stdio_client(PROBE) as (read, write), ClientSession(read, write) as session:
        await session.initialize()                       # protocol handshake

        for t in (await session.list_tools()).tools:     # what a host would hand to the LLM
            print("tool       :", t.name)
            print("description:", t.description)
            print("inputSchema:", json.dumps(t.inputSchema))

        ok = await session.call_tool("parse_to_iso", {"text": "16 February 2024", "fmt": "%d %B %Y"})
        print("\ngood call  :", ok.content[0].text, "| isError:", ok.isError)

        bad = await session.call_tool("parse_to_iso", {"text": "16 February 2024", "fmt": "%Y-%m-%d"})
        print("bad call   :", bad.content[0].text, "| isError:", bad.isError)


await raw_probe()

tool       : parse_to_iso
description: Parse `text` with the strptime format `fmt` and return the date as YYYY-MM-DD.
inputSchema: {"properties": {"text": {"title": "Text", "type": "string"}, "fmt": {"title": "Fmt", "type": "string"}}, "required": ["text", "fmt"], "title": "parse_to_isoArguments", "type": "object"}

good call  : 2024-02-16 | isError: False
bad call   : Error executing tool parse_to_iso: time data '16 February 2024' does not match format '%Y-%m-%d' | isError: True


**Observations:**
- The input schema is generated from `text: str, fmt: str`, and the description is the docstring. Nothing is written by hand.
- **The bad call failed on the server, but it didn't crash the client.** The server raised a
  `ValueError`, yet `call_tool` returned normally: there's no traceback, and the cell carried on to print.
  The failure came back *as data*, with the error message as the result text and the flag
  `isError: True`. If the LLM is the caller, it can read that message and retry with a different argument.
- This level works, but it only speaks MCP, and Gemini doesn't know what MCP is. To let Gemini use this
  tool, we would have to write the glue by hand: (1) turn the tool listing above into the tool format
  Gemini accepts, and (2) whenever Gemini asks for `parse_to_iso(...)`, pass that request on as
  `session.call_tool(...)` and hand the result back.

### 2.5 stdio + `langchain-mcp-adapters`

Same server, same transport. This is the **alternative** to §2.4. Internally the adapter opens the same kind of `ClientSession`, lists the tools, and wraps each one as a LangChain tool that sends a `call_tool` to the server when it runs. We don't touch `ClientSession` ourselves.

How the §2.4 steps map onto the adapter:

| Manual MCP client (§2.4) | LangChain adapter (§2.5) |
|---|---|
| `StdioServerParameters` | `MultiServerMCPClient` |
| `stdio_client()` | internally handled |
| `ClientSession()` | internally handled |
| `session.list_tools()` | `get_tools()` |
| `session.call_tool()` | `tool.ainvoke()` |

In [5]:
from langchain_mcp_adapters.client import MultiServerMCPClient

stdio_client_lc = MultiServerMCPClient({
    "probe": {"transport": "stdio", "command": sys.executable, "args": ["mcp_servers/probe_server.py"]},
})
stdio_tool = (await stdio_client_lc.get_tools())[0]
print(type(stdio_tool).__name__, "|", stdio_tool.name, "|", stdio_tool.args)

for text in ("16 February 2024", "15 February 2008"):
    t0 = time.perf_counter()
    out = await stdio_tool.ainvoke({"text": text, "fmt": "%d %B %Y"})
    print(f"{text!r:>20} -> {out}   ({time.perf_counter() - t0:.2f}s)")

StructuredTool | parse_to_iso | {'text': {'title': 'Text', 'type': 'string'}, 'fmt': {'title': 'Fmt', 'type': 'string'}}


  '16 February 2024' -> [{'type': 'text', 'text': '2024-02-16', 'id': 'lc_ec231bed-fb13-4d45-8f81-7d852bc826ba'}]   (0.26s)


  '15 February 2008' -> [{'type': 'text', 'text': '2008-02-15', 'id': 'lc_0189f30d-c4e2-42dc-ab0a-5bed0de8bce0'}]   (0.24s)


**Observations:**
- The result is a regular LangChain `StructuredTool`, so `llm.bind_tools([...])` works on it like any other tool.
- **Every call starts a fresh server subprocess** (the adapter's default is stateless: new session per
  call), and the ~0.25s per call above is almost all that startup cost. For two dates that doesn't matter. For an agent that makes many tool calls we shuold keep one session open (`client.session(...)` + `load_mcp_tools`) so every new call will go back to the same running process --> pay the startup cost once instead of on every call.

### 2.6 Streamable HTTP + `langchain-mcp-adapters`

Same server again, but over HTTP on localhost. What's different is the process lifecycle: the server
has to be started separately, the client has to wait until the port is up, and the server has to be
stopped afterwards.

In [6]:
# Ask the OS for a free port instead of hard-coding one to prevent collision
with socket.socket() as s:
    s.bind(("127.0.0.1", 0))
    port = s.getsockname()[1]

server = subprocess.Popen([sys.executable, "mcp_servers/probe_server.py", "http", str(port)],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
try:
    for _ in range(50):                                   # wait until the port accepts connections
        try:
            socket.create_connection(("127.0.0.1", port), timeout=0.1).close()
            break
        except OSError:
            time.sleep(0.1)

    http_client_lc = MultiServerMCPClient({
        "probe": {"transport": "streamable_http", "url": f"http://127.0.0.1:{port}/mcp"},
    })
    http_tool = (await http_client_lc.get_tools())[0]
    for text in ("16 February 2024", "15 February 2008"):
        t0 = time.perf_counter()
        out = await http_tool.ainvoke({"text": text, "fmt": "%d %B %Y"})
        print(f"{text!r:>20} -> {out}   ({time.perf_counter() - t0:.2f}s)")
finally:
    server.terminate()
    server.wait()

  '16 February 2024' -> [{'type': 'text', 'text': '2024-02-16', 'id': 'lc_9e3fadf6-b52a-4d88-8113-217f2ca34445'}]   (0.01s)
  '15 February 2008' -> [{'type': 'text', 'text': '2008-02-15', 'id': 'lc_70f9ed52-b3a1-44ad-8b73-9615fd41b315'}]   (0.01s)


**Observations:**
- The tool looks identical to the stdio one. Transport is invisible above the client.
- Calls take ~0.01s, against ~0.25s over stdio, because the server process is already running.
- The cost is operational: a port to pick, a readiness wait, and a process to clean up. That's worth
  paying when several clients share one server or it runs on another machine. 

## 3. Summary and decision

| Option | Transport | What it takes | Fit for this task |
|---|---|---|---|
| A. `FastMCP` server + raw `ClientSession` (§2.4) | stdio | hand-convert schemas to Gemini declarations and route calls yourself | most transparent, but re-implements the adapter |
| **B. `FastMCP` server + `langchain-mcp-adapters` (§2.5)** | stdio | one server file; the client starts it | local and portless, the most literal "local MCP"; tools become LangChain tools for `bind_tools` |
| C. `FastMCP` server + `langchain-mcp-adapters` (§2.6) | Streamable HTTP | start the server, manage a port, stop it | built for shared or remote servers; overhead for the current task |

**Decisions:**

| Decision | Chosen | Why |
|---|---|---|
| Transport | **stdio** | Streamable HTTP adds work that only pays off when a server is shared: §2.6 had to start the server as its own process, pick a free port, wait until that port accepted connections, and stop the process afterwards. What HTTP buys is many clients sharing one server, or a server on another machine, and faster calls (~0.01s against ~0.25s) because the process stays up. None of that applies here: one notebook, one client, two dates, so the time saved is under half a second. With stdio the client starts and stops the server itself. |
| Server library | **official SDK's `FastMCP`** | Builds tool schemas from type hints and docstrings. The standalone `fastmcp` package's extras aren't needed. |
| Client | **`langchain-mcp-adapters`** | MCP tools become `StructuredTool`s, so they bind to the same Gemini model through LangChain, as in Part 1 (option A would re-implement this). |
| Tool interface | **`normalize_date(text)`, text only** | Split the work: the LLM decides *which* date on the page matters (the judgement), and the tool converts it (the mechanical part). Formats live in code, so the LLM can't pass a wrong one. |

## 4. The datetime MCP server

Design notes:
- **A fixed list of written-out formats, not a fuzzy parser.** Both dates in the document are
  `D Month YYYY` (§1). A few common unambiguous variants are accepted too. A fuzzy parser is built to
  guess, and a confident wrong guess is exactly what the tool is meant to prevent.
- **No numeric `DD/MM/YYYY` or `MM/DD/YYYY`.** `03/04/2024` could be 3 April or 4 March, so it's rejected rather than guessed.
- **Bad input raises.** §2.4 showed a raised error reaches the client as `isError` with the message,
  so the message says what to pass instead, and the LLM can retry.
- **The docstring is the LLM-facing prompt.** It says to pass one date, not a whole sentence or a range.

In [7]:
%%writefile mcp_servers/datetime_server.py
"""Local MCP server with one datetime tool: normalise a written date to ISO 8601 (YYYY-MM-DD)."""
from datetime import datetime

from mcp.server.fastmcp import FastMCP

mcp = FastMCP("datetime")

# Unambiguous written-out formats only. Numeric forms like 03/04/2024 are left out on purpose:
# day-first vs month-first can't be told apart, and a silent wrong guess is worse than an error.
FORMATS = ["%d %B %Y", "%d %b %Y", "%B %d, %Y", "%b %d, %Y", "%B %d %Y", "%Y-%m-%d"]


@mcp.tool()
def normalize_date(text: str) -> str:
    """Convert ONE calendar date written in text (e.g. '16 February 2024') to ISO format YYYY-MM-DD.

    Pass only the date itself - not the surrounding sentence, and not a date range.
    """
    cleaned = " ".join(text.split()).strip(" .,;:")
    for fmt in FORMATS:
        try:
            return datetime.strptime(cleaned, fmt).date().isoformat()
        except ValueError:
            pass
    raise ValueError(f"Could not read {text!r} as a single date. Pass just one date, e.g. '16 February 2024'.")


if __name__ == "__main__":
    mcp.run(transport="stdio")

Overwriting mcp_servers/datetime_server.py


### 4.1 Test the tool before giving it to an LLM

This runs through the same path the LLM will use (stdio → adapter → `StructuredTool`), with no LLM
involved, so any later failure can be pinned on either the tool or the prompt. The cases cover the
document's two dates, the accepted variants, and inputs that should be refused.

In [8]:
DATETIME_SERVER = {
    "datetime": {"transport": "stdio", "command": sys.executable, "args": ["mcp_servers/datetime_server.py"]},
}
mcp_tools = await MultiServerMCPClient(DATETIME_SERVER).get_tools()
normalize_date = next(t for t in mcp_tools if t.name == "normalize_date")
print("LLM will see:", normalize_date.description, "\n")

cases = [
    "16 February 2024",               # page 1
    "15 February 2008.",              # page 36, with the sentence's full stop
    "Feb 16, 2024", "2024-02-16",     # accepted variants
    "03/04/2024",                     # ambiguous numeric - should be refused
    "1 April 2024 to 31 March 2025",  # a range, not one date - should be refused
]
for text in cases:
    out = await normalize_date.ainvoke({"text": text})   # a list of content blocks
    print(f"{text!r:>34} -> {out[0]['text']}")

LLM will see: Convert ONE calendar date written in text (e.g. '16 February 2024') to ISO format YYYY-MM-DD.

Pass only the date itself - not the surrounding sentence, and not a date range.
 



                '16 February 2024' -> 2024-02-16


               '15 February 2008.' -> 2008-02-15


                    'Feb 16, 2024' -> 2024-02-16


                      '2024-02-16' -> 2024-02-16


                      '03/04/2024' -> Error executing tool normalize_date: Could not read '03/04/2024' as a single date. Pass just one date, e.g. '16 February 2024'.


   '1 April 2024 to 31 March 2025' -> Error executing tool normalize_date: Could not read '1 April 2024 to 31 March 2025' as a single date. Pass just one date, e.g. '16 February 2024'.


**Observations:**
- Both document dates normalise correctly (`2024-02-16`, `2008-02-15`), and the trailing full stop is handled.
- The ambiguous numeric date and the date range are refused with the hint message, as designed.
- **The adapter doesn't raise on a tool error.** The server raised `ValueError`, but the adapter returns
  the `Error executing tool ...` message as ordinary text content, just like a successful result. For
  the LLM this is what we want: it reads the hint and can retry. For our code, it means **a failed
  normalisation can't be detected by catching an exception**. The next step has to check that the tool
  output really is a `YYYY-MM-DD` date before using it.

## 5. Extraction + tool calling

**What is reused from Part 1:** the same model (`gemini-3.1-flash-lite`, temperature 0), the same
pdfplumber pages, the same page scoping (one call per item, each seeing only its own page, as Part 1's `FIELD_PAGES` does), the same `<page number="N">`
tags, and the same retry settings.

**Design decisions:**

| Decision | Why |
|---|---|
| **A new system prompt, not Part 1's** | Part 1's grounding rules carry over (use only the supplied pages, cite the page, quote verbatim). Its financial-figure rules (financial year and basis, negative figures in parentheses, rebuilding table headers) don't apply to dates, so they're left out. |
| **One call per date, each seeing only its own page** | Part 1 §2.3 found that a model shown several pages at once took a figure from the wrong page, while a page left out of the context can't be used at all. It also keeps the answer schema to a single `ExtractedDate`, with no list and no label saying which date it is: the code already knows which date each call is for. The cost is one tool-calling loop per date instead of one for both. |
| **Tool instructions go in the system prompt, not the user message** | A user asks for the dates; *how* they get normalised is the system's job, and rule 4 says so. Leaving the tool out of the user message also means the run shows the model choosing `normalize_date` itself, rather than being told to. |
| **A hand-written tool loop** | The loop is: the LLM replies → if it asked for a tool, run it on the MCP server and send the result back → repeat. It's a few lines, and it prints every tool call, which is what this part is about. |
| **The answer schema is offered as one more tool**, and the model must call *some* tool every turn (`tool_choice="any"`) | Calling `ExtractedDate` is how the model hands in its final answer, so the loop ends on a clear signal and no extra call is needed to turn a text reply into structured output. |
| **`normalized_date` is checked against the tool log** | The prompt says to copy the tool's output. The check in §5.1 confirms that each date really came from a successful tool call in that date's own run, so a date the LLM wrote itself is caught. |
| **At most 5 turns** | A safety stop, so a model that keeps calling tools can't run up API calls on your key. |

In [9]:
from typing import Literal

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from pydantic import BaseModel, Field

load_dotenv(Path(".env"))

MODEL = "gemini-3.1-flash-lite"   # same model as Part 1
RETRY = dict(stop_after_attempt=5, wait_exponential_jitter=True,   # same retry settings as Part 1
             exponential_jitter_params={"initial": 4, "max": 60, "exp_base": 2, "jitter": 2})
MAX_TURNS = 5

llm = ChatGoogleGenerativeAI(model=MODEL, temperature=0)


def context_for(page_numbers: list[int]) -> str:
    """Same as Part 1: render selected pages as tagged blocks so the model can cite page numbers."""
    return "\n\n".join(f'<page number="{n}">\n{PAGES[n].strip()}\n</page>' for n in sorted(page_numbers))


EXTRACT_SYSTEM_V1 = """You are a meticulous document analyst working with Singapore Government Budget documents.

Rules:
1. Use ONLY what the supplied pages state. The pages below are the sole source of truth.
2. Each page is wrapped in <page number="N"> tags. N is the document's printed page number - use it for source_page.
3. original_text must be copied verbatim from the page text.
4. Every date must be normalised with the normalize_date tool. Never write an ISO date yourself.
5. Pass the tool only the date itself (e.g. '16 February 2024'), not the sentence around it.
6. If the tool returns an error, read the message and call it again with corrected input.
7. When the requested date has been normalised, call ExtractedDate with the final answer."""

EXTRACT_USER = """Find this date in the page below of the FY2024 Analysis of Revenue and Expenditure (Singapore Ministry of Finance): {request}

{context}"""

# What to find, and the page it is on - like Part 1's FIELD_PAGES. Each date gets its own call that sees only its own page.
DATE_REQUESTS = {
    "distribution_date": ("The document's distribution date.", [1]),
    "estate_duty_date":  ("The date relating to estate duty.", [36]),
}


class ExtractedDate(BaseModel):
    """Final answer: the requested date, with its normalised form taken from the normalize_date tool."""
    source_page: int = Field(description="The printed page number the date was read from.")
    original_text: str = Field(description="The sentence containing the date, copied verbatim from the page.")
    date_text: str = Field(description="The date exactly as written in original_text, e.g. '16 February 2024'.")
    normalized_date: str = Field(description="The normalize_date tool's output for date_text. Copy it exactly; never write it yourself.")


print("Extraction prompt v1 and schema ready.")

Extraction prompt v1 and schema ready.


In [10]:
def tool_text(message) -> str:
    """The adapter's ToolMessage content is a list of content blocks (§4.1) - join the text parts."""
    return "".join(block["text"] for block in message.content if block.get("type") == "text")


async def extract_date(system_prompt: str, request: str, pages: list[int]) -> tuple[ExtractedDate, list[dict]]:
    """LLM tool-calling loop for ONE date over the MCP datetime tool. Returns the answer and a log of every tool call."""
    # Connect to the local MCP server (stdio) and get its tools as LangChain tools
    mcp_tools = {t.name: t for t in await MultiServerMCPClient(DATETIME_SERVER).get_tools()}

    # Offer Gemini two "tools": the real normalize_date, and ExtractedDate - calling that one hands in the answer.
    # tool_choice="any" makes it call one of them every turn instead of replying in plain text.
    llm_with_tools = llm.bind_tools([*mcp_tools.values(), ExtractedDate], tool_choice="any").with_retry(**RETRY)

    # The conversation so far: rules, then the request plus only this date's page
    messages = [SystemMessage(system_prompt),
                HumanMessage(EXTRACT_USER.format(request=request, context=context_for(pages)))]
    tool_log = []

    for turn in range(1, MAX_TURNS + 1):
        # 1. Ask Gemini what to do next, given everything so far
        reply = await llm_with_tools.ainvoke(messages)
        messages.append(reply)

        # 2. Handle each tool call in the reply
        final = None
        for call in reply.tool_calls:
            if call["name"] == "ExtractedDate":
                # Nothing to run: the call's arguments ARE the final answer
                final = ExtractedDate(**call["args"])
                continue

            # A real tool: run it on the MCP server, and add the result to the conversation for Gemini's next turn
            result = await mcp_tools[call["name"]].ainvoke(call)   # returns a ToolMessage
            messages.append(result)
            tool_log.append({"turn": turn, "args": call["args"], "output": tool_text(result), "status": result.status})
            print(f"  turn {turn}: {call['name']}({call['args']}) -> {tool_text(result)!r} [{result.status}]")

        # 3. Stop once the answer is in; otherwise loop back to Gemini with the tool results
        if final:
            print(f"  turn {turn}: ExtractedDate (final answer)")
            return final, tool_log

    # Safety stop, so a model that never hands in an answer can't keep spending API calls
    raise RuntimeError(f"No final answer after {MAX_TURNS} turns")


async def run_extraction(system_prompt: str) -> tuple[dict, dict]:
    """One tool-calling loop per date, each seeing only its own page. Used unchanged for every prompt version."""
    extracted, tool_logs = {}, {}   # both keyed by date name, e.g. "distribution_date"
    for name, (request, pages) in DATE_REQUESTS.items():
        print(f"{name} (page {pages})")
        extracted[name], tool_logs[name] = await extract_date(system_prompt, request, pages)
    return extracted, tool_logs


extracted_v1, tool_logs_v1 = await run_extraction(EXTRACT_SYSTEM_V1)

# Show each extracted date with its evidence, then the step 1 output the task asks for: the list of normalised dates
print()
for name, d in extracted_v1.items():
    print(name, json.dumps(d.model_dump(), indent=2))
print("\nNormalised dates:", [d.normalized_date for d in extracted_v1.values()])

distribution_date (page [1])


  turn 1: normalize_date({'text': '16 February 2024'}) -> '2024-02-16' [success]


  turn 2: ExtractedDate (final answer)
estate_duty_date (page [36])


  turn 1: normalize_date({'text': '15 February 2008'}) -> '2008-02-15' [success]


  turn 2: ExtractedDate (final answer)

distribution_date {
  "source_page": 1,
  "original_text": "Distributed on Budget Day: 16 February 2024",
  "date_text": "16 February 2024",
  "normalized_date": "2024-02-16"
}
estate_duty_date {
  "source_page": 36,
  "original_text": "Estate Duty does not apply to a person who dies after 15 February 2008.",
  "date_text": "15 February 2008",
  "normalized_date": "2008-02-15"
}

Normalised dates: ['2024-02-16', '2008-02-15']


### 5.1 Check the extraction

Expected values were read by hand from the PDF in §1. Each date is checked four ways:
- **correct date:** it matches the hand-read value.
- **ISO format:** it is `YYYY-MM-DD`.
- **came from the tool:** a successful `normalize_date` call returned it, so the LLM didn't write it itself.
- **quote on cited page:** `original_text` appears on the page it cites, ignoring line breaks. This is the same idea as Part 1's provenance check.

In [11]:
EXPECTED_DATES = {"distribution_date": "2024-02-16", "estate_duty_date": "2008-02-15"}   # read by hand, §1


def flat(text: str) -> str:
    """Collapse line breaks and repeated spaces, so a quote can match across lines."""
    return " ".join(text.split())


def quote_on_page(quote: str, page_no: int) -> bool:
    """True if the quote appears on the cited page, ignoring line breaks."""
    return flat(quote) in flat(PAGES.get(page_no, ""))


def check_extraction(extracted: dict[str, ExtractedDate], tool_logs: dict[str, list[dict]]) -> None:
    for name, d in extracted.items():
        tool_outputs = {entry["output"] for entry in tool_logs[name] if entry["status"] == "success"}
        checks = {
            "correct date":        d.normalized_date == EXPECTED_DATES[name],
            "ISO format":          bool(re.fullmatch(r"\d{4}-\d{2}-\d{2}", d.normalized_date)),
            "came from the tool":  d.normalized_date in tool_outputs,
            "quote on cited page": quote_on_page(d.original_text, d.source_page),
        }
        print(name)
        for check, ok in checks.items():
            print(f"    {'PASS' if ok else 'FAIL'}  {check}")


check_extraction(extracted_v1, tool_logs_v1)

distribution_date
    PASS  correct date
    PASS  ISO format
    PASS  came from the tool
    PASS  quote on cited page
estate_duty_date
    PASS  correct date
    PASS  ISO format
    PASS  came from the tool
    FAIL  quote on cited page


In [12]:
# Why did the estate-duty quote fail? Print the page 36 lines the sentence sits on.
lines = PAGES[36].splitlines()
for i, line in enumerate(lines):
    if "Estate Duty does not apply" in line:
        print("\n".join(lines[i:i + 2]))

her death. Estate Duty does not apply to a Endowment Fund, ElderCare Endowment Fund,
person who dies after 15 February 2008. Lifelong Learning Endowment Fund, and


**Observations:**
- **The model chose the tool on its own.** The user message doesn't mention `normalize_date`; only the
  system prompt's rule 4 does, and it was still called for both dates.
- **The LLM did the tool calling as designed, the same way for each date.** In turn 1 it called
  `normalize_date` once, passing only the date (rule 5). In turn 2 it handed in `ExtractedDate`. That's 2
  Gemini calls per date, 4 in total. No tool call failed, so rule 6 (read the error and retry) was never tested.
- **Both normalised dates are correct, and each came from that date's own tool call:** `['2024-02-16', '2008-02-15']`.
- **The one FAIL is the estate-duty quote, and the check is what's wrong, not the quote.** The lines
  printed above show that pdfplumber reads page 36's two glossary columns straight across. Text from the
  neighbouring column ("Endowment Fund, ElderCare Endowment Fund,") lands in the middle of the sentence.
  The model returned the sentence as it reads on the page, which isn't a substring of that extracted
  text, so an exact-match check can't find it. §5.2 fixes the check.

### 5.2 Fix the quote check: read a two-column page one column at a time

pdfplumber can crop a page to a rectangle (`page.crop(...)`) and extract text from just that area, on
whichever pages we choose. Page 36's columns can be read separately only if there's a clean gap between
them, so check that first.

In [13]:
# Is there a clean gap between page 36's columns? Check before splitting the page at half its width.
with pdfplumber.open(PDF_PATH) as pdf:
    page36 = pdf.pages[35]
    mid = page36.width / 2
    words = page36.extract_words()

print("words crossing the middle:", [w["text"] for w in words if w["x0"] < mid < w["x1"]])
print(f"left column ends at x={max(w['x1'] for w in words if w['x1'] < mid):.1f} | "
      f"middle at x={mid:.1f} | right column starts at x={min(w['x0'] for w in words if w['x0'] > mid):.1f}")

words crossing the middle: []
left column ends at x=285.6 | middle at x=297.7 | right column starts at x=324.1


No word crosses the middle of the page, and the middle falls inside the gap between the two columns,
so splitting page 36 at half its width won't cut through any text.

**Design decisions:**

| Decision | Why |
|---|---|
| **Only page 36 is split** (`TWO_COLUMN_PAGES = {36}`) | It's the only two-column page this task cites. Page 1 already passes as it is. Detecting columns automatically on every page would be more code than two pages need. |
| **Split at half the page width** | Checked above: no word crosses that line. |
| **A quote passes if it appears on the whole page *or* in either column** | Text that isn't in the columns, such as page 1's lines, still matches the way it did before. |
| **Only the check changes; nothing is re-extracted** | The LLM's answer was already right. `check_extraction` and `extracted_v1` are reused as they are, with only `quote_on_page` redefined, so this costs no Gemini calls. |

In [14]:
TWO_COLUMN_PAGES = {36}


def columns_of(page_no: int) -> list[str]:
    """Text of the left and right halves of a page, each extracted on its own."""
    with pdfplumber.open(PDF_PATH) as pdf:
        page = pdf.pages[page_no - 1]
        mid = page.width / 2
        return [page.crop((0, 0, mid, page.height)).extract_text() or "",
                page.crop((mid, 0, page.width, page.height)).extract_text() or ""]


def quote_on_page(quote: str, page_no: int) -> bool:
    """v2: on a two-column page, also look for the quote in each column."""
    texts = [PAGES.get(page_no, "")] + (columns_of(page_no) if page_no in TWO_COLUMN_PAGES else [])
    return any(flat(quote) in flat(t) for t in texts)


check_extraction(extracted_v1, tool_logs_v1)   # same check, same LLM output - only quote_on_page changed

distribution_date
    PASS  correct date
    PASS  ISO format
    PASS  came from the tool
    PASS  quote on cited page
estate_duty_date
    PASS  correct date
    PASS  ISO format
    PASS  came from the tool
    PASS  quote on cited page


**Observation:** all four checks now pass for both dates. The extracted answer is exactly the
same as in §5.1; only the quote check changed.

## 6. Reasoning over the normalised dates

**Design decisions:**

| Decision | Why |
|---|---|
| **Input is §5's output, not the pages** | The task says to classify "using the normalized dates from the previous step". |
| **One call per date; code carries `original_text` and `normalized_date` through** | Same as §5: the schema is a single `DateStatus`, with no list. The model returns only `reasoning` and `status`, so it can't alter the text or date it was given, because the final output takes those straight from §5. Each date is also judged on its own, without the other date in view. |
| **`original_text` is passed along with each date** | `normalize_date` returns a single date, so whether that date starts a *period* can only be read from the original wording. That wording is what separates Ongoing from Expired. |
| **Few-shot examples, none taken from the document** | An example using this document's dates would give the model the answers. There's one example each for Expired and Upcoming, and two for Ongoing: a period with an end date, and a rule with no end date. |
| **Reasoning before status** | The same lesson as Part 1 §3.2: the model writes out its comparison before committing to a label. |
| **The LLM does the date comparison** | The task asks the LLM to reason. Code only checks the result afterwards. |

**Assumption: a past date that starts a rule still in effect on the reference date is Ongoing, not Expired.**
The task defines Expired as "the date has already passed" and Ongoing as "the date refers to a period
that is currently active". A date like 15 February 2008 fits both readings: the date itself is before
2024-01-01, but the rule it starts ("Estate Duty does not apply to a person who dies after 15 February
2008") still applies on 2024-01-01. This notebook takes the Ongoing reading, because the date matters
as the start of something still active, not as a moment that has passed.
- **Whether a rule is still in effect is judged from the document's wording,** here a rule stated in the
  present tense with no end date, not from outside knowledge.
- **This rule is written into `CLASSIFY_SYSTEM_V1` and one few-shot example (the next cell),** and §6.1 expects
  Ongoing for the estate-duty date. Under the other reading, that date would be Expired.

In [ ]:
# Prompt v1 for classification: the reference date, what each status means, then worked examples.
# None of the examples use this document's dates, so they don't give the model the answers.
CLASSIFY_SYSTEM_V1 = """You classify dates taken from a Singapore Government Budget document against a reference date.

Reference date: 2024-01-01

The date comes with the original_text it was taken from. Give it exactly one status:
- Expired: the date is a one-off event, or the end of a period, before the reference date.
- Upcoming: the date is after the reference date.
- Ongoing: the date refers to a period that is active on the reference date. This includes a period, rule or policy that started before the reference date and is still in effect.

Use original_text to decide whether the date is a one-off event or the start of a period. A date in the past is not automatically Expired.

Examples:

original_text: "The scheme closed on 30 June 2023."
normalized_date: 2023-06-30
reasoning: A one-off closing date. 2023-06-30 is before 2024-01-01 and nothing continues after it.
status: Expired

original_text: "Applications open on 1 March 2024."
normalized_date: 2024-03-01
reasoning: 2024-03-01 is after 2024-01-01.
status: Upcoming

original_text: "The financial year runs from 1 April 2023 to 31 March 2024."
normalized_date: 2023-04-01
reasoning: The date starts a period that ends on 2024-03-31, and 2024-01-01 falls inside it.
status: Ongoing

original_text: "The levy does not apply to vehicles registered after 1 July 2015."
normalized_date: 2015-07-01
reasoning: The date starts a rule with no end date, stated in the present tense, so the rule is still in effect on 2024-01-01.
status: Ongoing"""

In [ ]:
class DateStatus(BaseModel):
    """The verdict for ONE date. original_text and normalized_date aren't repeated here - code carries them over from §5."""
    # reasoning comes before status, so the model writes out its comparison before choosing a label
    reasoning: str = Field(description="Say whether the date is a one-off event or the start of a period, then compare it with 2024-01-01.")
    status: Literal["Expired", "Upcoming", "Ongoing"]


async def classify_date(system_prompt: str, date: ExtractedDate) -> DateStatus:
    """One structured-output call for ONE date extracted in §5."""
    # The model only sees the text and the normalised date - not the PDF pages
    given = {"original_text": date.original_text, "normalized_date": date.normalized_date}

    # No tools here: the answer comes back directly as a DateStatus
    chain = llm.with_structured_output(DateStatus).with_retry(**RETRY)
    return await chain.ainvoke([SystemMessage(system_prompt),
                                HumanMessage("Classify this date:\n" + json.dumps(given, indent=2))])


async def run_classification(system_prompt: str, extracted: dict[str, ExtractedDate]) -> dict[str, DateStatus]:
    """One call per date, like §5. Used unchanged for every prompt version."""
    classified = {}   # keyed by date name, e.g. "distribution_date"
    for name, date in extracted.items():
        classified[name] = await classify_date(system_prompt, date)
    return classified

In [15]:
# Classify each date from §5 (one Gemini call per date)
classified_v1 = await run_classification(CLASSIFY_SYSTEM_V1, extracted_v1)

# Show each verdict with the model's reasoning
for name, verdict in classified_v1.items():
    print(f"{extracted_v1[name].normalized_date}  {verdict.status:<9} {verdict.reasoning}")

# Final output in the task's sample format: text and date come from §5, only the status comes from this step
final_output = [{"original_text": extracted_v1[name].original_text,
                 "normalized_date": extracted_v1[name].normalized_date,
                 "status": verdict.status}
                for name, verdict in classified_v1.items()]
print("\n" + json.dumps(final_output, indent=2))

Direct use of automatic function calling (AFC) in AsyncModels.generate_content is not recommended. Instead, we recommend to use AFC in AsyncChat.send_message. Similarly, direct use of AFC in AsyncModels.generate_content_stream is not recommended. Instead, we recommend to use AFC in AsyncChat.send_message_stream.


2024-02-16  Upcoming  The date 2024-02-16 represents a one-off event occurring after the reference date of 2024-01-01.
2008-02-15  Ongoing   The date marks the start of a policy change that remains in effect as of 2024-01-01, as the exemption for deaths after this date is still current.

[
  {
    "original_text": "Distributed on Budget Day: 16 February 2024",
    "normalized_date": "2024-02-16",
    "status": "Upcoming"
  },
  {
    "original_text": "Estate Duty does not apply to a person who dies after 15 February 2008.",
    "normalized_date": "2008-02-15",
    "status": "Ongoing"
  }
]


### 6.1 Check the classification

In [16]:
EXPECTED_STATUS = {"distribution_date": "Upcoming", "estate_duty_date": "Ongoing"}   # Ongoing: design decision in §6


def check_classification(extracted: dict[str, ExtractedDate], classified: dict[str, DateStatus]) -> None:
    for name, verdict in classified.items():
        expected = EXPECTED_STATUS[name]
        print(f"{'PASS' if verdict.status == expected else 'FAIL'}  {name} ({extracted[name].normalized_date}): "
              f"got {verdict.status}, expected {expected}")


check_classification(extracted_v1, classified_v1)

PASS  distribution_date (2024-02-16): got Upcoming, expected Upcoming
PASS  estate_duty_date (2008-02-15): got Ongoing, expected Ongoing


**Observations:**
- **Both statuses match:** 2024-02-16 is Upcoming, and 2008-02-15 is Ongoing. That's one Gemini call per date, 2 in total.
- **Only the status comes from this step.** The `original_text` and `normalized_date` in the final output
  are §5's values, carried over by code, so they match §5 exactly.
- **The reasoning shows `original_text` was used.** The distribution date is read as a one-off event after
  the reference date. The estate-duty date is read as the start of a policy that is still in effect
  ("the exemption for deaths after this date is still current"). That distinction can't be made from the date alone.
- As noted in §6, Ongoing follows the rule written into the prompt and the fourth few-shot example. The
  model applied the decision rather than reaching it on its own.
- The "automatic function calling (AFC)" warning above is printed by the Google client library, not by
  this notebook. Both calls still returned valid results.